# Set up

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import json

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_validate
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import(
    LogisticRegression,
    Ridge,
    Lasso
)
from sklearn.metrics import(
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    PrecisionRecallDisplay,
    ConfusionMatrixDisplay
)


## Data

In [ ]:
df = pd.read_parquet("../data/brfss2025_il_clean.parquet")
with open("../data/brfss2025_il_clean_columns.json") as f:
    meta = json.load(f)
    
cat_predictors, num_predictors, bin_predictors, predictor_groups = meta["cat_predictors"], meta["num_predictors"],meta["bin_predictors"], meta["predictor_groups"]
predictors = cat_predictors + num_predictors + bin_predictors
group_predictors = [c for cols in predictor_groups.values() for c in cols]

num_set, cat_set, bin_set = set(num_predictors), set(cat_predictors), set(bin_predictors)
# No column in more than one type list
assert len(predictors) == len(set(predictors)), {c for c in predictors if predictors.count(c) > 1}

# Type lists and groups dict contain the same names
assert set(predictors) == set(group_predictors), set(predictors) ^ set(group_predictors)

# All predictors in data
assert set(predictors) <= set(df.columns), set(predictors) - set(df.columns)

train = df[df["split"] == "train"]
test = df[df["split"] == "test"]
X_train, y_train = train[predictors], train[meta["target"]]
X_test, y_test = test[predictors], test[meta["target"]]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["roc_auc", "average_precision", "recall", "f1"]

print(len(X_train) + len(X_test))
print(len(y_train) + len(y_test))
print(len(df))


## Functions

In [ ]:
def pick(group=None, kind=None):
    """
    Predictors filtered by group and/or type
    
    group: "demographic", "behavior", "health", or None for all groups
    kind: "num", "cat", "bin" or None for both
    """
    cols = predictor_groups[group] if group else predictors
    if kind == "num":
        return [c for c in cols if c in num_set]
    if kind == "cat":
        return [c for c in cols if c in cat_set]
    if kind == "bin":
        return [c for c in cols if c in bin_set]
    return list(cols)

def uspstf_flag(frame, medians):
    f = frame[["bmi", "age"]].fillna(medians)
    return ((f["bmi"] >= 25) & f["age"].between(35,70)).astype(int)


# USPSTF recommendation

In [ ]:
meds = train[["bmi", "age"]].median()           # Use training median even for test set

uspstf_train = uspstf_flag(train, meds)
uspstf_train_flag_rate = uspstf_train.mean()

uspstf_test = uspstf_flag(test, meds)
uspstf_test_flag_rate = uspstf_test.mean()

print("USPSTF recommendation results\n\nTraining")
print(f"flag rate:\t{uspstf_train_flag_rate:.5f}")
print(f"recall:\t\t{recall_score(y_train, uspstf_train):.5f}")
print(f"\nTesting\nflag rate:\t{uspstf_test_flag_rate:.5f}")
print(f"recall:\t\t{recall_score(y_test, uspstf_test):.5f}")


# Baseline - Logistic regression 

## Preprocessor

In [ ]:
preprocessor = ColumnTransformer([
    ("cat", make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OneHotEncoder(drop="first", handle_unknown="ignore")),
    cat_predictors),
    ("num", make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler()),
    num_predictors),
    ("bin", SimpleImputer(strategy="most_frequent"), bin_predictors),
])

- 19% of rows have a missing value, experiment with models that drop these rows

## Base Model

In [ ]:
logreg = Pipeline([
    ("pre", clone(preprocessor)),
    ("logreg", LogisticRegression(C=np.inf, class_weight="balanced")),
])

logreg_cv_scores = cross_validate(
    logreg,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
)

print("CV metrics:")
for m in scoring:
    te, tr = logreg_cv_scores[f"test_{m}"], logreg_cv_scores[f"train_{m}"]
    print(f"{m:<18} CV {te.mean():.4f} \u00B1 {te.std():.4f}\ttrain: {tr.mean():.4f}")

## Coefficients

In [ ]:
logreg.fit(X_train, y_train)
coefs = pd.Series(
    logreg.named_steps["logreg"].coef_[0],
    index=logreg.named_steps["pre"].get_feature_names_out(),
)
odds_ratios = np.exp(coefs).sort_values(ascending=False)
display(odds_ratios)

## Threshold

In [ ]:
logreg.fit(X_train, y_train)
prob =logreg.predict_proba(X_train)[:,1]
threshold_grid = np.linspace(0.05, 0.95, 20).round(3)

rows = []

for thr in threshold_grid:
    pred_thr = (prob >= thr).astype(int)
    cm = confusion_matrix(y_train, pred_thr)
    tn, fp, fn, tp = cm.ravel()

    rows.append({
        "threshold": thr,
        "recall": recall_score(y_train, pred_thr),
        "f1": f1_score(y_train, pred_thr),
    })

metrics = pd.DataFrame(rows)
metrics = metrics.set_index("threshold")
display(metrics.round(3))


# Regularized logistic regression

## Preprocessor - same as baseline logreg

In [ ]:
preprocessor = ColumnTransformer([
    ("cat", make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OneHotEncoder(drop="first", handle_unknown="ignore")),
    cat_predictors),
    ("num", make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler()),
    num_predictors),
    ("bin", SimpleImputer(strategy="most_frequent"), bin_predictors),
])